# 05 — Validação das matrizes de conectividade

Este notebook valida as matrizes de conectividade funcional calculadas no Notebook 04.

### Verificações realizadas

- existência da matriz de conectividade;
- existência e correspondência do arquivo de rs-fMRI;
- dimensão da matriz;
- simetria;
- diagonal igual a 1;
- ausência de `NaN` e `Inf`;
- valores dentro do intervalo esperado de correlação [-1, 1];
- número de volumes (timepoints) das séries temporais;
- consistência da quantidade de participantes;
- consistência das 190 ROIs.

> A amostra final já foi definida no Notebook 04: 709 participantes, sendo 453 controles e 256 participantes com TDAH. O objetivo deste notebook é validar a integridade dos dados, e não redefinir a amostra.

In [5]:

# 1. Carregar a amostra final


from pathlib import Path
import sys
import numpy as np
import pandas as pd


# Localizar raiz do projeto


def encontrar_raiz_projeto():
    candidatos = [Path.cwd(), *Path.cwd().parents]

    for caminho in candidatos:
        if (caminho / "config.py").exists():
            return caminho.resolve()

    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto TCC_TDAH."
    )


RAIZ = encontrar_raiz_projeto()

if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import config


# Verificar tabela de participantes


if not config.ARQ_PARTICIPANTES.exists():
    raise FileNotFoundError(
        "Tabela de participantes não encontrada:\n"
        f"{config.ARQ_PARTICIPANTES}"
    )

df = pd.read_csv(
    config.ARQ_PARTICIPANTES,
    dtype={
        "ID_padronizado": str,
        "subject_id": str
    }
)

print("=" * 60)
print("VALIDAÇÃO — AMOSTRA FINAL")
print("=" * 60)

print(
    f"\nParticipantes na tabela: "
    f"{len(df):,}"
)

print(
    f"Participantes esperados: "
    f"709"
)


# Verificar tamanho da amostra


if len(df) != 709:
    print(
        "\n⚠ ATENÇÃO: a tabela não contém exatamente "
        "709 participantes."
    )
else:
    print(
        "\n✓ A tabela contém os 709 participantes "
        "da amostra final."
    )


# Verificar colunas necessárias


colunas_necessarias = [
    "ID_padronizado",
    "site",
    "caminho",
    "classe"
]

faltantes = [
    c for c in colunas_necessarias
    if c not in df.columns
]

if faltantes:
    raise KeyError(
        "Colunas obrigatórias ausentes:\n"
        + "\n".join(f"  - {c}" for c in faltantes)
    )


# Verificar IDs duplicados


duplicados = df["ID_padronizado"].duplicated().sum()

print(
    f"\nIDs duplicados: {duplicados}"
)

if duplicados > 0:
    raise ValueError(
        "Existem IDs duplicados na tabela de participantes."
    )

print("\nDistribuição das classes:")

display(
    df["classe"]
    .value_counts()
    .sort_index()
)

VALIDAÇÃO — AMOSTRA FINAL

Participantes na tabela: 709
Participantes esperados: 709

✓ A tabela contém os 709 participantes da amostra final.

IDs duplicados: 0

Distribuição das classes:


classe
0    453
1    256
Name: count, dtype: int64

In [6]:

# 2. Validar matrizes de conectividade


linhas = []

for _, r in df.iterrows():

    participante_id = r["ID_padronizado"]

    arquivo_fc = (
        config.CONNECTIVITY /
        f"FC_{participante_id}.npy"
    )

    reg = {
        "ID": participante_id,
        "site": r["site"],
        "classe": r["classe"],
        "fc_existe": arquivo_fc.exists(),
        "rsfmri_existe": Path(r["caminho"]).exists(),
    }


    # Validar arquivo da matriz FC


    if arquivo_fc.exists():

        fc = np.load(
            arquivo_fc,
            allow_pickle=False
        )

        reg.update({

            "shape_ok": (
                fc.shape ==
                (
                    config.N_ROIS,
                    config.N_ROIS
                )
            ),

            "simetrica": bool(
                np.allclose(
                    fc,
                    fc.T,
                    atol=1e-6
                )
            ),

            "diag_ok": bool(
                np.allclose(
                    np.diag(fc),
                    1.0,
                    atol=1e-6
                )
            ),

            "finita": bool(
                np.isfinite(fc).all()
            ),

            "faixa_ok": bool(
                fc.min() >= -1.0001
                and
                fc.max() <= 1.0001
            ),

            "shape": str(fc.shape),

            "min": float(fc.min()),

            "max": float(fc.max())

        })

    else:

        reg.update({

            "shape_ok": False,
            "simetrica": False,
            "diag_ok": False,
            "finita": False,
            "faixa_ok": False,
            "shape": None,
            "min": None,
            "max": None
        })


    # Validar número de volumes da série temporal


    caminho_rsfmri = Path(r["caminho"])

    if caminho_rsfmri.exists():

        try:

            ts = pd.read_csv(
                caminho_rsfmri,
                sep="\t"
            )

            reg["n_volumes"] = len(ts)

            roi_cols = [
                c for c in ts.columns
                if str(c).startswith("Mean_")
            ]

            reg["n_rois_series"] = len(roi_cols)

            reg["roi_count_ok"] = (
                len(roi_cols) == config.N_ROIS
            )

            reg["series_finita"] = bool(
                np.isfinite(
                    ts[roi_cols]
                    .apply(
                        pd.to_numeric,
                        errors="coerce"
                    )
                    .values
                ).all()
            )

        except Exception as erro:

            reg["n_volumes"] = None
            reg["n_rois_series"] = None
            reg["roi_count_ok"] = False
            reg["series_finita"] = False
            reg["erro_series"] = str(erro)

    else:

        reg["n_volumes"] = None
        reg["n_rois_series"] = None
        reg["roi_count_ok"] = False
        reg["series_finita"] = False

    linhas.append(reg)


val = pd.DataFrame(linhas)


# Resumo das verificações


checks = [
    "fc_existe",
    "rsfmri_existe",
    "shape_ok",
    "simetrica",
    "diag_ok",
    "finita",
    "faixa_ok",
    "roi_count_ok",
    "series_finita"
]

resultado_checks = (
    val[checks]
    .fillna(False)
    .all()
)

print("=" * 60)
print("RESULTADO DAS VALIDAÇÕES")
print("=" * 60)

display(resultado_checks)

falhas = (
    ~val[checks]
    .fillna(False)
    .all(axis=1)
)

print(
    f"\nParticipantes com alguma falha: "
    f"{int(falhas.sum())}"
)

if falhas.sum() > 0:

    print("\nParticipantes com falhas:")

    display(
        val.loc[
            falhas,
            [
                "ID",
                "site",
                "classe"
            ] + checks
        ]
    )

else:

    print(
        "\n✓ Todos os 709 participantes "
        "passaram em todas as verificações."
    )

RESULTADO DAS VALIDAÇÕES


fc_existe        True
rsfmri_existe    True
shape_ok         True
simetrica        True
diag_ok          True
finita           True
faixa_ok         True
roi_count_ok     True
series_finita    True
dtype: bool


Participantes com alguma falha: 0

✓ Todos os 709 participantes passaram em todas as verificações.


In [7]:

# 3. Resumo dos volumes das séries temporais


print("=" * 60)
print("NÚMERO DE VOLUMES POR SÉRIE TEMPORAL")
print("=" * 60)

print("\nResumo geral:")

display(
    val["n_volumes"].describe()
)

print("\nVolumes por site:")

display(
    val.groupby("site")["n_volumes"]
    .agg(
        quantidade="count",
        minimo="min",
        mediana="median",
        maximo="max"
    )
)

print("\nROIs encontradas nas séries temporais:")

display(
    val["n_rois_series"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\n✓ A quantidade de volumes é apresentada "
      "como informação descritiva.")

NÚMERO DE VOLUMES POR SÉRIE TEMPORAL

Resumo geral:


count    709.000000
mean     177.789845
std       52.477863
min       72.000000
25%      148.000000
50%      172.000000
75%      232.000000
max      257.000000
Name: n_volumes, dtype: float64


Volumes por site:


,quantidade,minimo,mediana,maximo
site,,,,
KKI,80,120,120.0,148
NYU,190,172,172.0,172
NeuroIMAGE,41,257,257.0,257
OHSU,72,72,74.0,74
Peking_1,82,232,232.0,232
Peking_2,67,232,232.0,232
Peking_3,42,232,232.0,232
Pittsburgh,88,192,192.0,192
WashU,47,72,128.0,129



ROIs encontradas nas séries temporais:


n_rois_series
190    709
Name: count, dtype: int64


✓ A quantidade de volumes é apresentada como informação descritiva.


## Interpretação dos volumes

O número de volumes (timepoints) é apresentado de forma descritiva para verificar a consistência das séries temporais entre os participantes e sites.

Não será aplicado neste notebook um ponto de corte arbitrário para excluir participantes com base exclusivamente no número de volumes, pois os dados já foram submetidos ao pipeline Athena.

Eventuais diferenças entre sites serão consideradas na interpretação e, posteriormente, na estratégia de validação dos modelos de Machine Learning.

In [8]:

# 5. Salvar relatório de validação


arquivo_validacao = (
    config.PROCESSED /
    "validacao.csv"
)

val.to_csv(
    arquivo_validacao,
    index=False
)

print("=" * 60)
print("VALIDAÇÃO CONCLUÍDA")
print("=" * 60)

print(
    f"\nRelatório salvo em:"
    f"\n{arquivo_validacao}"
)

print(
    f"\nParticipantes avaliados: "
    f"{len(val):,}"
)

print(
    f"Matrizes esperadas: "
    f"{len(df):,}"
)

print(
    f"ROIs esperadas por matriz: "
    f"{config.N_ROIS}"
)

print(
    f"Conexões únicas por participante: "
    f"{config.N_CONNECTIONS:,}"
)

VALIDAÇÃO CONCLUÍDA

Relatório salvo em:
C:\Users\Maria Eduarda\Downloads\TCC_TDAH\data\processed\validacao.csv

Participantes avaliados: 709
Matrizes esperadas: 709
ROIs esperadas por matriz: 190
Conexões únicas por participante: 17,955
